# 06 Conversational Interface

Extracted from `FP_NewsBot2_ksm_GroupName_ITAI2373.ipynb` (original outputs kept). Module D definitions and the intent-accuracy test. The live demo needs the trained `bot` object - run 07_System_Integration (or the full notebook) for the end-to-end chat demo.

Upload `BBC News Train.csv` (in `data/raw/`) to Colab before running.

---
## 🅳 Module D: Conversational Interface
We built a simple chatbot that understands questions in normal English. It has three parts:
1. **Intent classifier** – keyword and regex rules that decide what the user wants (show, summarize, sentiment, search, entity, topics, translate, classify, export, follow-up).
2. **Query processor** – splits multi-part questions ("show tech articles **and** summarize article 3") and keeps **context** (it remembers the last list and last category, so "more" and "what about the sentiment" work).
3. **Response generator** – calls the right NewsBot module and writes the answer.

We did not use a big language model, so it works offline and it is easy to explain.

In [21]:
"""Module D: rule-based intent classification (keywords + regex)."""
import re

CATEGORIES = ["business", "entertainment", "politics", "sport", "tech"]
SYNONYMS = {"sports": "sport", "technology": "tech", "political": "politics",
            "finance": "business", "economy": "business", "movies": "entertainment",
            "music": "entertainment", "film": "entertainment"}


def find_category(text):
    words = re.findall(r"[a-z]+", text.lower())
    for w in words:
        if w in CATEGORIES:
            return w
        if w in SYNONYMS:
            return SYNONYMS[w]
    return None


def find_sentiment_filter(text):
    q = text.lower()
    if re.search(r"\b(positive|good|happy)\b", q):
        return "positive"
    if re.search(r"\b(negative|bad|sad)\b", q):
        return "negative"
    return None


def detect_intent(query):
    q = query.lower().strip()
    num = re.search(r"\b(\d+)\b", q)
    if re.search(r"\b(hi|hello|hey)\b", q):
        return "greeting", {}
    if re.search(r"\b(help|what can you do)\b", q):
        return "help", {}
    if "summar" in q and num:
        return "summarize", {"index": int(num.group(1))}
    if "sentiment" in q or "mood" in q or "feel" in q:
        return "sentiment", {"category": find_category(q), "index": int(num.group(1)) if num else None}
    if "topic" in q:
        return "topics", {}
    if re.search(r"\b(translate)\b", q):
        m = re.search(r"translate\s+(?:article\s+)?(\d+)\s+(?:to|into)\s+(\w+)", q)
        if m:
            return "translate", {"index": int(m.group(1)), "lang": m.group(2)}
    if re.search(r"\b(classify|categor)", q):
        text = re.sub(r"^.*?(classify|categorize)\s*:?\s*", "", query, flags=re.I)
        return "classify", {"text": text}
    m = re.search(r"\b(?:who is|tell me about|news about|mentions of)\s+(.+)$", q)
    if m and not re.search(r"\barticles?\b", q):
        return "entity", {"name": m.group(1).strip(" ?.")}
    if re.search(r"\b(more|another|next|show more)\b", q) and len(q.split()) <= 4:
        return "follow_up", {}
    if re.search(r"\b(report|export)\b", q):
        return "export", {}
    if re.search(r"\b(find|search|about|related)\b", q):
        m = re.search(r"(?:about|for|on|related to)\s+(.+)$", q)
        return "search", {"query": m.group(1) if m else q}
    if re.search(r"\b(show|list|give)\b", q) and find_category(q):
        return "show_category", {"category": find_category(q), "sentiment": find_sentiment_filter(q)}
    if find_category(q):
        return "show_category", {"category": find_category(q), "sentiment": find_sentiment_filter(q)}
    return "unknown", {}


def split_multi_part(query):
    """Handle multi-part questions like 'show tech articles and summarize article 3'."""
    parts = re.split(r"\s+(?:and then|and also|then|;)\s+|\s+and\s+(?=(?:show|summar|find|what|translate|classify|list))", query, flags=re.I)
    return [p for p in parts if p and p.strip()]

In [22]:
"""Module D: turns an intent + slots into a friendly answer using the NewsBot components."""

HELP_TEXT = """I can help with:
  - "show positive sports articles"
  - "summarize article 5"
  - "what's the sentiment of business news"
  - "find articles about elections"
  - "who is tony blair"           (entity lookup)
  - "what are the main topics"
  - "translate article 3 to spanish"
  - "classify: <some news text>"
  - "more" / "next"               (follow-up on the last list)
  - "export report"
You can also combine: "show tech articles and summarize article 3"."""

LANG_CODES = {"spanish": "es", "french": "fr", "german": "de", "italian": "it",
              "portuguese": "pt", "arabic": "ar", "chinese": "zh-CN"}


class ResponseGenerator:
    def __init__(self, bot):
        self.bot = bot

    def _list(self, sub, title):
        lines = [f"[{i}] {t[:85]}..." for i, t in zip(sub.index, sub["content"])]
        return title + "\n" + "\n".join(lines)

    def respond(self, intent, s, ctx):
        b, df = self.bot, self.bot.df
        if intent == "greeting":
            return "Hi! I'm NewsBot 2.0. Type 'help' to see what I can do."
        if intent == "help":
            return HELP_TEXT
        if intent == "show_category":
            sub = df[df["category"] == s["category"]]
            if s.get("sentiment"):
                sub = sub[b.sentiment_labels.loc[sub.index] == s["sentiment"]]
            ctx["last_results"], ctx["offset"] = list(sub.index), 5
            ctx["last_category"] = s["category"]
            mood = f"{s['sentiment']} " if s.get("sentiment") else ""
            return self._list(sub.head(5), f"Here are 5 of {len(sub)} {mood}{s['category']} articles:")
        if intent == "follow_up":
            res, off = ctx.get("last_results", []), ctx.get("offset", 0)
            if not res:
                return "There is nothing to continue yet. Try 'show tech articles' first."
            ctx["offset"] = off + 5
            nxt = res[off:off + 5]
            return self._list(df.loc[nxt], "Next results:") if nxt else "No more results."
        if intent == "summarize":
            i = s["index"]
            if i not in df.index:
                return f"Sorry, article {i} does not exist (0-{len(df) - 1})."
            ctx["last_article"] = i
            return f"Summary of article {i} ({df.loc[i, 'category']}):\n" + b.summarize(df.loc[i, "content"])
        if intent == "sentiment":
            if s.get("index") is not None and s["index"] in df.index:
                r = b.sentiment.analyze(df.loc[s["index"], "content"])
                return f"Article {s['index']} is {r['label']} (compound={r['compound']:.2f})."
            cat = s.get("category") or ctx.get("last_category")
            mask = df["category"] == cat if cat else df["category"].notna()
            scores = b.sentiment_scores[mask]
            return (f"Sentiment of {cat or 'all'} news: average compound {scores.mean():.2f}, "
                    f"{(scores >= 0.05).mean() * 100:.0f}% positive articles.")
        if intent == "search":
            q = b.searcher.expand_query(s["query"])
            hits = b.search(q, top_k=5)
            ctx["last_results"], ctx["offset"] = [i for i, _ in hits], 5
            if not hits:
                return "No matching articles found."
            return f"Top articles about '{s['query']}' (expanded query: '{q}'):\n" + "\n".join(
                f"[{i}] ({df.loc[i, 'category']}, score {sc:.2f}) {df.loc[i, 'content'][:75]}..." for i, sc in hits)
        if intent == "entity":
            name = s["name"].lower()
            mask = df["content"].str.lower().str.contains(name, regex=False)
            if not mask.any():
                return f"I could not find '{s['name']}' in the articles."
            cats = df[mask]["category"].value_counts().to_dict()
            mood = b.sentiment_scores[mask].mean()
            ctx["last_results"], ctx["offset"] = list(df[mask].index), 5
            return (f"'{s['name']}' appears in {int(mask.sum())} articles {cats}. "
                    f"Average tone of those articles: {mood:.2f}. Say 'more' to list them.")
        if intent == "topics":
            return "Main topics:\n" + "\n".join(f"{k}: {', '.join(v[:8])}" for k, v in b.topics().items())
        if intent == "translate":
            i = s["index"]
            if i not in df.index:
                return f"Sorry, article {i} does not exist."
            code = LANG_CODES.get(s["lang"], s["lang"])
            text, ok = translate(b.summarize(df.loc[i, "content"], 2), target=code, source="en")
            return text if ok else "Translation service not available right now (offline?). Original:\n" + text
        if intent == "classify":
            if not s["text"].strip():
                return "Please give me some text, e.g. 'classify: the team won the cup final'."
            r = b.analyze_article(s["text"])
            return f"Category: {r['category']} ({r['confidence']:.0%}), sentiment: {r['sentiment']}."
        if intent == "export":
            path = b.export_report()
            return f"Report saved to {path}"
        return "Sorry, I did not understand. Type 'help' for examples."

In [23]:
"""Module D: main entry for natural-language questions. Keeps conversation state (context)."""
import logging

log = logging.getLogger(__name__)


class QueryProcessor:
    def __init__(self, bot):
        self.responder = ResponseGenerator(bot)
        self.context = {}      # remembers last results, last category, last article
        self.history = []

    def process(self, query):
        answers = []
        for part in split_multi_part(query):
            intent, slots = detect_intent(part)
            log.info("query=%r intent=%s slots=%s", part, intent, slots)
            try:
                ans = self.responder.respond(intent, slots, self.context)
            except Exception as e:  # never crash the chat
                ans = f"Something went wrong ({type(e).__name__}). Please try another question."
            self.history.append({"query": part, "intent": intent, "answer": ans})
            answers.append(ans)
        return "\n\n".join(answers)

In [26]:
# Intent accuracy on a small hand-labelled test set we wrote ourselves
intent_tests = [
    ('show sports articles', 'show_category'), ('list business news', 'show_category'), ('technology news please', 'show_category'),
    ('summarize article 10', 'summarize'), ('can you summarize article 42', 'summarize'),
    ("what's the sentiment of politics news", 'sentiment'), ('how do people feel about sport', 'sentiment'),
    ('sentiment of article 7', 'sentiment'), ('find articles about oil prices', 'search'), ('search for football transfers', 'search'),
    ('articles related to music awards', 'search'), ('who is gordon brown', 'entity'), ('tell me about microsoft', 'entity'),
    ('what are the main topics', 'topics'), ('translate article 2 to french', 'translate'), ('classify: the band released a new album', 'classify'),
    ('hi', 'greeting'), ('help', 'help'), ('more', 'follow_up'), ('export report', 'export'),
]
pred = [detect_intent(q)[0] for q, _ in intent_tests]
intent_acc = np.mean([p == t for p, (_, t) in zip(pred, intent_tests)])
print(f'Intent accuracy: {intent_acc:.0%} on {len(intent_tests)} test questions')
for (q, t), p in zip(intent_tests, pred):
    if p != t:
        print('  wrong:', q, '-> got', p, 'expected', t)

Intent accuracy: 100% on 20 test questions


In [27]:
# Try it yourself! (type 'quit' to stop). Commented out so 'Run all' does not wait for input.
# while True:
#     q = input('You: ')
#     if q.lower() in ['quit', 'exit']:
#         break
#     print('NewsBot:', bot.ask(q))